# Download and store data

This notebook contains information on downloading the Quandl Wiki stock prices and a few other sources that we use throughout the book. 

## Imports & Settings

In [1]:
# %pip install tables

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from pathlib import Path
import requests
from io import BytesIO, StringIO
from zipfile import ZipFile, BadZipFile

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

pd.set_option('display.expand_frame_repr', False)

## Set Data Store path

Modify path if you would like to store the data elsewhere and change the notebooks accordingly

In [4]:
DATA_STORE = Path('assets.h5')

## Quandl Wiki Prices

> Quandl has been [acuqired by NASDAQ](https://www.nasdaq.com/about/press-center/nasdaq-acquires-quandl-advance-use-alternative-data) in late 2018. In 2021, NASDAQ [integrated Quandl's data platform](https://data.nasdaq.com/). Free US equity data is still available under a [new URL](https://data.nasdaq.com/databases/WIKIP/documentation), subject to the limitations mentioned below.

[NASDAQ](https://data.nasdaq.com/) makes available a [dataset](/home/stefan/drive/machine-learning-for-trading/data/create_datasets.ipynb) with stock prices, dividends and splits for 3000 US publicly-traded companies. Prior to its acquisition (April 11, 2018), Quandl announced the end of community support (updates). The historical data are useful as a first step towards demonstrating the application of the machine learning solutions, just ensure you design and test your own algorithms using current, professional-grade data.

1. Follow the instructions to create a free [NASDAQ account](https://data.nasdaq.com/sign-up)
2. [Download](https://data.nasdaq.com/tables/WIKIP/WIKI-PRICES/export) the entire WIKI/PRICES data
3. Extract the .zip file,
4. Move to this directory and rename to wiki_prices.csv
5. Run the below code to store in fast HDF format (see [Chapter 02 on Market & Fundamental Data](../02_market_and_fundamental_data) for details).

In [5]:
df = (pd.read_csv('wiki_prices.csv',
                 parse_dates=['date'],
                 index_col=['date', 'ticker'])
     .sort_index())
# print(df.info(show_counts=True))
print(df.info(show_counts=True))
with pd.HDFStore(DATA_STORE) as store:
    store.put('quandl/wiki/prices', df)

<class 'pandas.DataFrame'>
MultiIndex: 15389314 entries, (Timestamp('1962-01-02 00:00:00'), 'ARNC') to (Timestamp('2018-03-27 00:00:00'), 'ZUMZ')
Data columns (total 12 columns):
 #   Column       Non-Null Count     Dtype  
---  ------       --------------     -----  
 0   open         15388776 non-null  float64
 1   high         15389259 non-null  float64
 2   low          15389259 non-null  float64
 3   close        15389313 non-null  float64
 4   volume       15389314 non-null  float64
 5   ex-dividend  15389314 non-null  float64
 6   split_ratio  15389313 non-null  float64
 7   adj_open     15388776 non-null  float64
 8   adj_high     15389259 non-null  float64
 9   adj_low      15389259 non-null  float64
 10  adj_close    15389313 non-null  float64
 11  adj_volume   15389314 non-null  float64
dtypes: float64(12)
memory usage: 1.4+ GB
None


### Wiki Prices Metadata

> QUANDL used to make some stock meta data be available on its website; I'm making the file available to allow readers to run some examples in the book:

Instead of using the QUANDL API, load the file `wiki_stocks.csv` as described and store in HDF5 format.

In [6]:
df = pd.read_csv('wiki_stocks.csv')
# no longer needed
# df = pd.concat([df.loc[:, 'code'].str.strip(),
#                 df.loc[:, 'name'].str.split('(', expand=True)[0].str.strip().to_frame('name')], axis=1)

print(df.info(show_counts=True))
with pd.HDFStore(DATA_STORE) as store:
    store.put('quandl/wiki/stocks', df)

<class 'pandas.DataFrame'>
RangeIndex: 3199 entries, 0 to 3198
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   code    3199 non-null   str  
 1   name    3199 non-null   str  
dtypes: str(2)
memory usage: 50.1 KB
None


## S&P 500 Prices

The following code downloads historical S&P 500 prices from FRED (only last 10 years of daily data is freely available)

In [7]:
start = pd.Timestamp('2009-01-01')
fred_url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500'

df = (pd.read_csv(fred_url, parse_dates=['observation_date'])
      .rename(columns={'observation_date': 'date', 'SP500': 'close'})
      .set_index('date'))
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df = df.loc[start:].dropna()
print(df.info())
with pd.HDFStore(DATA_STORE) as store:
    store.put('sp500/fred', df)

<class 'pandas.DataFrame'>
DatetimeIndex: 2513 entries, 2016-05-16 to 2026-05-13
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   close   2513 non-null   float64
dtypes: float64(1)
memory usage: 39.3 KB
None


Alternatively, download S&P500 data from [stooq.com](https://stooq.com/q/?s=%5Espx&c=1d&t=l&a=lg&b=0); at the time of writing the data was available since 1789. You can switch from Polish to English on the lower right-hand side.

We store the data from 1950-2020:

In [8]:
sp500_stooq = (pd.read_csv('^spx_d.csv', index_col=0,
                     parse_dates=True).loc['1950':'2019'].rename(columns=str.lower))
print(sp500_stooq.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 17700 entries, 1950-01-03 to 2019-12-31
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    17700 non-null  float64
 1   high    17700 non-null  float64
 2   low     17700 non-null  float64
 3   close   17700 non-null  float64
 4   volume  17700 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 829.7 KB
None


In [9]:
with pd.HDFStore(DATA_STORE) as store:
    store.put('sp500/stooq', sp500_stooq)

### S&P 500 Constituents

The following code downloads the current S&P 500 constituents from [Wikipedia](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies).

In [10]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ),
}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()
df = pd.read_html(StringIO(r.text), header=0)[0]

In [11]:
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [12]:
# df.columns = ['ticker', 'name', 'sec_filings', 'gics_sector', 'gics_sub_industry',
#               'location', 'first_added', 'cik', 'founded']
# df = df.drop('sec_filings', axis=1).set_index('ticker')

df.columns = ['ticker', 'name', 'gics_sector', 'gics_sub_industry',
              'location', 'first_added', 'cik', 'founded']
df = df.set_index('ticker')

In [13]:
print(df.info())

<class 'pandas.DataFrame'>
Index: 503 entries, MMM to ZTS
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   name               503 non-null    str  
 1   gics_sector        503 non-null    str  
 2   gics_sub_industry  503 non-null    str  
 3   location           503 non-null    str  
 4   first_added        503 non-null    str  
 5   cik                503 non-null    int64
 6   founded            503 non-null    str  
dtypes: int64(1), str(6)
memory usage: 31.4+ KB
None


In [14]:
with pd.HDFStore(DATA_STORE) as store:
    store.put('sp500/stocks', df)

## Metadata on US-traded companies

The following downloads several attributes for [companies](https://www.nasdaq.com/market-activity/stocks/screener) traded on NASDAQ, AMEX and NYSE.

> **Note:** The old `render=download` screener URLs on `nasdaq.com` return an HTML app page, not a CSV, so `pd.read_csv(url)` will not work. The same data is available from **`https://api.nasdaq.com/api/screener/stocks`** as JSON (`download=true`, `exchange=nasdaq`, `nyse`, or `amex`). The next cell uses that API. Review Nasdaq's terms of use and avoid aggressive polling.


In [15]:
# Nasdaq screener JSON API (exchange slug must be lowercase)
NASDAQ_SCREENER_URL = 'https://api.nasdaq.com/api/screener/stocks'
NASDAQ_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'application/json',
    'Origin': 'https://www.nasdaq.com',
    'Referer': 'https://www.nasdaq.com/',
}


def fetch_nasdaq_screener(exchange: str) -> pd.DataFrame:
    """exchange: 'nasdaq', 'nyse', or 'amex'."""
    r = requests.get(
        NASDAQ_SCREENER_URL,
        params={
            'tableonly': 'true',
            'limit': 25000,
            'offset': 0,
            'download': 'true',
            'exchange': exchange,
        },
        headers=NASDAQ_HEADERS,
        timeout=120,
    )
    r.raise_for_status()
    rows = r.json().get('data', {}).get('rows') or []
    return pd.DataFrame(rows)


In [16]:
# Company metadata: use local screener CSVs if all three exist; otherwise use the
# Nasdaq API from the cell above (run that cell first if you do not have CSV files).

file_paths = {
    'AMEX': 'AMEX.csv',
    'NASDAQ': 'NASDAQ.csv',
    'NYSE': 'NYSE.csv',
}

use_csv = all(Path(p).exists() for p in file_paths.values())

if use_csv:
    df = pd.concat(
        [pd.read_csv(path) for path in file_paths.values()]
    ).dropna(how='all', axis=1)
elif 'fetch_nasdaq_screener' in globals():
    # Fetch once per exchange, save CSVs next to this notebook for future offline runs.
    frames = []
    for ex, fname in (
        ('nasdaq', 'NASDAQ.csv'),
        ('nyse', 'NYSE.csv'),
        ('amex', 'AMEX.csv'),
    ):
        part = fetch_nasdaq_screener(ex)
        part.to_csv(fname, index=False)
        frames.append(part)
    df = pd.concat(frames, ignore_index=True).dropna(how='all', axis=1)
else:
    missing = [p for p in file_paths.values() if not Path(p).exists()]
    raise FileNotFoundError(
        'Missing '
        + ', '.join(missing)
        + '. Either run the Nasdaq API cell above (defines fetch_nasdaq_screener) '
        'or download AMEX/NASDAQ/NYSE screener CSVs into this folder.'
    )

df = df.rename(columns=str.lower).set_index('symbol').drop(
    'summary quote', axis=1, errors='ignore'
)
df = df[~df.index.duplicated()]

print(df.info())


<class 'pandas.DataFrame'>
Index: 7072 entries, AACB to ZONE
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   name       7072 non-null   str  
 1   lastsale   7072 non-null   str  
 2   netchange  7072 non-null   str  
 3   pctchange  7072 non-null   str  
 4   volume     7072 non-null   str  
 5   marketcap  7072 non-null   str  
 6   country    7072 non-null   str  
 7   ipoyear    7072 non-null   str  
 8   industry   7072 non-null   str  
 9   sector     7072 non-null   str  
 10  url        7072 non-null   str  
dtypes: str(11)
memory usage: 921.0+ KB
None


In [17]:
df = df.rename(columns={'market cap': 'marketcap'})
df.head()


,name,lastsale,netchange,pctchange,volume,marketcap,country,ipoyear,industry,sector,url
symbol,,,,,,,,,,,
AACB,Artius II Acquisition Inc. Class A Ordinary Sh...,$10.43,0.00,0.00%,30,0.00,United States,2025,,,/market-activity/stocks/aacb
AACBR,Artius II Acquisition Inc. Rights,$0.2776,0.00,0.00%,139,0.00,United States,2025,,,/market-activity/stocks/aacbr
AACG,ATA Creativity Global American Depositary Shares,$1.19,-0.0313,-2.563%,8796,51205696.00,China,2008,Other Consumer Services,Real Estate,/market-activity/stocks/aacg
AACI,Armada Acquisition Corp. III Class A Ordinary ...,$9.93,-0.01,-0.101%,206,0.00,United States,2026,,,/market-activity/stocks/aaci
AACIU,Armada Acquisition Corp. III Units,$10.08,0.00,0.00%,27,0.00,United States,2026,Blank Checks,Finance,/market-activity/stocks/aaciu


### Convert market cap information to numerical format

Market cap is provided as strings so we need to convert it to numerical format.

In [18]:
# mcap = df[['marketcap']].dropna()
# mcap['suffix'] = mcap['marketcap'].astype(str).str[-1]
# mcap.suffix.value_counts()

Keep only values with value units:

In [19]:
# mcap = mcap[mcap.suffix.str.endswith(('B', 'M'))]
# mcap.marketcap = pd.to_numeric(mcap['marketcap'].astype(str).str[:-1], errors='coerce')
# mcaps = {'M': 1e6, 'B': 1e9}
# for symbol, factor in mcaps.items():
#     mcap.loc[mcap.suffix == symbol, 'marketcap'] *= factor
# mcap.info()

In [20]:
# df['marketcap'] = mcap.marketcap
# df.marketcap.describe(percentiles=np.arange(.1, 1, .1).round(1)).apply(
#     lambda x: 'nan' if pd.isna(x) else f'{int(x):,d}'
# )


### Store result

The file `us_equities_meta_data.csv` contains a version of the data used for many of the examples. Load using 
```
df = pd.read_csv('us_equities_meta_data.csv')
```
and proceed to store in HDF5 format.

In [21]:
df = pd.read_csv('us_equities_meta_data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6834 entries, 0 to 6833
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ticker     6834 non-null   str    
 1   name       6834 non-null   str    
 2   lastsale   6718 non-null   float64
 3   marketcap  5766 non-null   float64
 4   ipoyear    3038 non-null   float64
 5   sector     5288 non-null   str    
 6   industry   5288 non-null   str    
dtypes: float64(3), str(4)
memory usage: 373.9 KB


In [22]:
with pd.HDFStore(DATA_STORE) as store:
    store.put('us_equities/stocks', df.set_index('ticker'))

## MNIST Data

In [23]:
mnist = fetch_openml('mnist_784', version=1)

In [24]:
print(mnist.DESCR)

**Author**: Yann LeCun, Corinna Cortes, Christopher J.C. Burges  
**Source**: [MNIST Website](http://yann.lecun.com/exdb/mnist/) - Date unknown  
**Please cite**:  

The MNIST database of handwritten digits with 784 features, raw data available at: http://yann.lecun.com/exdb/mnist/. It can be split in a training set of the first 60,000 examples, and a test set of 10,000 examples  

It is a subset of a larger set available from NIST. The digits have been size-normalized and centered in a fixed-size image. It is a good database for people who want to try learning techniques and pattern recognition methods on real-world data while spending minimal efforts on preprocessing and formatting. The original black and white (bilevel) images from NIST were size normalized to fit in a 20x20 pixel box while preserving their aspect ratio. The resulting images contain grey levels as a result of the anti-aliasing technique used by the normalization algorithm. the images were centered in a 28x28 image b

In [25]:
mnist.keys()

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])

In [26]:
mnist_path = Path('mnist')
if not mnist_path.exists():
    mnist_path.mkdir()

In [27]:
np.save(mnist_path / 'data', mnist.data.astype(np.uint8))
np.save(mnist_path / 'labels', mnist.target.astype(np.uint8))

## Fashion MNIST Image Data

We will use the Fashion MNIST image data created by [Zalando Research](https://github.com/zalandoresearch/fashion-mnist) for some demonstrations.

In [28]:
fashion_mnist = fetch_openml(name='Fashion-MNIST')

In [29]:
print(fashion_mnist.DESCR)

**Author**: Han Xiao, Kashif Rasul, Roland Vollgraf  
**Source**: [Zalando Research](https://github.com/zalandoresearch/fashion-mnist)  
**Please cite**: Han Xiao and Kashif Rasul and Roland Vollgraf, Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms, arXiv, cs.LG/1708.07747  

Fashion-MNIST is a dataset of Zalando's article images, consisting of a training set of 60,000 examples and a test set of 10,000 examples. Each example is a 28x28 grayscale image, associated with a label from 10 classes. Fashion-MNIST is intended to serve as a direct drop-in replacement for the original MNIST dataset for benchmarking machine learning algorithms. It shares the same image size and structure of training and testing splits. 

Raw data available at: https://github.com/zalandoresearch/fashion-mnist

### Target classes
Each training and test example is assigned to one of the following labels:
Label  Description  
0  T-shirt/top  
1  Trouser  
2  Pullover  
3  Dress  
4  

In [30]:
label_dict = {0: 'T-shirt/top',
              1: 'Trouser',
              2: 'Pullover',
              3: 'Dress',
              4: 'Coat',
              5: 'Sandal',
              6: 'Shirt',
              7: 'Sneaker',
              8: 'Bag',
              9: 'Ankle boot'}

In [31]:
fashion_path = Path('fashion_mnist')
if not fashion_path.exists():
    fashion_path.mkdir()

In [32]:
pd.Series(label_dict).to_csv(fashion_path / 'label_dict.csv', index=False, header=None)

In [33]:
np.save(fashion_path / 'data', fashion_mnist.data.astype(np.uint8))
np.save(fashion_path / 'labels', fashion_mnist.target.astype(np.uint8))


## Bond Price Indexes

The following code downloads several bond indexes from the Federal Reserve Economic Data service ([FRED](https://fred.stlouisfed.org/))

> Warning: Unfortunately, most of this data has been [recently removed](https://news.research.stlouisfed.org/2022/01/ice-benchmark-administration-ltd-iba-data-to-be-removed-from-fred/) from the FRED service. It is not important for the examples in the book, so you can just ignore this.

In [34]:
# securities = {'BAMLCC0A0CMTRIV'   : 'US Corp Master TRI',
#               'BAMLHYH0A0HYM2TRIV': 'US High Yield TRI',
#               'BAMLEMCBPITRIV'    : 'Emerging Markets Corporate Plus TRI',
#               'GOLDAMGBD228NLBM'  : 'Gold (London, USD)',
#               'DGS10'             : '10-Year Treasury CMR',
#               }

# df = web.DataReader(name=list(securities.keys()), data_source='fred', start=2000)
# df = df.rename(columns=securities).dropna(how='all').resample('B').mean()

# with pd.HDFStore(DATA_STORE) as store:
#     store.put('fred/assets', df)